# Day 25: Exceptions and Defensive Input

**Today's question:** how do we stop bad input or a missing file from **crashing** our program?

Before running a cell marked 🤔, predict what it will show.

## 1. Warm-up: errors we have already met

| Code | Error | First seen |
|------|-------|-----------|
| `int("abc")` | `ValueError` | Day 03 |
| `"word"[10]` | `IndexError` | Day 05 |
| `prices["juice"]` | `KeyError` | Day 08 |
| `10 / 0` | `ZeroDivisionError` | Day 03 |
| `open("missing.txt")` | `FileNotFoundError` | Day 23 |

These are called **exceptions**. A crash is simply an exception that **nobody handled**.

In [ ]:
age = int("twelve")
print("This line never runs")

The last line names the **type** (`ValueError`) and gives a **message**. Today we learn to catch it.

## 2. `try` / `except`

In [ ]:
text = "twelve"

try:
    number = int(text)
    print("You typed", number)
except ValueError:
    print(f"'{text}' is not a whole number")

print("The program keeps running")

How Python runs it:

1. Run the `try` block line by line.
2. If a line raises a `ValueError`, **skip the rest of the `try` block** and run the `except ValueError` block.
3. Either way, continue after the whole `try`/`except`.

🤔 **Predict:** what is printed when the text is a valid number?

In [ ]:
text = "12"

try:
    number = int(text)
    print("You typed", number)
except ValueError:
    print(f"'{text}' is not a whole number")

print("The program keeps running")

## 3. Catch **specific** exceptions

Different problems deserve different messages. One `try` can have several `except` blocks.

In [ ]:
def split_bill(total_text, people_text):
    """Print each person's share of a bill, with a helpful message for bad input."""
    try:
        total = float(total_text)
        people = int(people_text)
        share = total / people
        print(f"Each person pays {share:.2f}")
    except ValueError:
        print("Please type numbers only.")
    except ZeroDivisionError:
        print("There must be at least one person.")

🤔 **Predict** the message for each call.

In [ ]:
split_bill("90", "4")
split_bill("90", "four")
split_bill("90", "0")

The same pattern for a dictionary lookup and a missing file:

In [ ]:
prices = {"tea": 1.5, "coffee": 2.25}
order = "juice"

try:
    print(f"{order}: {prices[order]}")
except KeyError:
    print(f"Sorry, we do not sell {order}.")

In [ ]:
try:
    with open("data/marks_2025.txt", encoding="utf-8") as marks_file:
        text = marks_file.read()
except FileNotFoundError:
    print("No marks file for 2025 yet.")

`except ValueError as error:` gives the exception a name, so we can show Python's own message:

In [ ]:
try:
    quantity = int("4.5")
except ValueError as error:
    print("Problem:", error)

## 4. `else` and `finally`

- `else`: runs only if the `try` block raised **nothing**.
- `finally`: runs **always**, error or not (good for "always do this" steps).

In [ ]:
def show_first_line(path):
    """Print the first line of a file, or a message if the file is missing."""
    try:
        with open(path, encoding="utf-8") as text_file:
            first_line = text_file.read().split("\n")[0]
    except FileNotFoundError:
        print(f"{path}: not found")
    else:
        print(f"{path}: {first_line}")
    finally:
        print(f"Finished checking {path}")

In [ ]:
show_first_line("data/marks.txt")
show_first_line("data/marks_2025.txt")

Why `else` instead of putting the `print` inside `try`? Keep the `try` block **small**: only the line that can fail.
Then an unexpected bug in the other lines is not mistaken for "file not found".

## 5. Retry loops: asking until the input is valid

On Day 13 we used `.isdigit()`, which rejects `"12.5"` and `"-3"`. With `try`/`except` we accept **anything
`float()` accepts** and ask again otherwise.

In [ ]:
def get_number(prompt):
    """Ask until the user types a valid number, then return it as a float."""
    while True:
        text = input(prompt)
        try:
            return float(text)
        except ValueError:
            print(f"'{text}' is not a number. Please try again.")

🤔 **Predict:** the user types `twelve`, then presses Enter without typing, then types `12.5`. How many error
messages appear?

In [ ]:
price = get_number("Price: ")
print("Price with tax:", round(price * 1.1, 2))

`return` inside the loop ends the function, and with it the `while True` loop. No `break` needed.

A range check on top of the conversion:

In [ ]:
def get_int_in_range(prompt, low, high):
    """Ask until the user types a whole number from low to high, then return it."""
    while True:
        text = input(prompt)
        try:
            number = int(text)
        except ValueError:
            print("Please type a whole number.")
            continue
        if number < low or number > high:
            print(f"Please choose a number from {low} to {high}.")
        else:
            return number

In [ ]:
rating = get_int_in_range("Rate today's lesson (1-5): ", 1, 5)
print("Thanks for rating", rating)

## 6. `raise`: errors from your own functions

Our own functions can refuse bad values with a **clear message** instead of returning nonsense.

In [ ]:
def percentage(part, whole):
    """Return part as a percentage of whole. whole must be greater than 0."""
    if whole <= 0:
        raise ValueError(f"whole must be greater than 0, got {whole}")
    return part / whole * 100


print(percentage(45, 60))

In [ ]:
percentage(5, 0)

Compare with Python's own message for `5 / 0`: `ZeroDivisionError: division by zero`. Ours says **which**
value was wrong and **what** was expected. And `percentage(5, -10)` would otherwise silently return `-50.0`.

The **caller** decides what to do with the error:

In [ ]:
try:
    print(percentage(5, 0))
except ValueError as error:
    print("Could not calculate:", error)

Why `raise` and not `print` inside the function? The function cannot know what the caller wants: show a message,
ask again, skip a line of a file… Raising hands the decision to the caller.

## 7. Why a bare `except:` is a bad idea

🤔 **Predict:** the input `"42"` is perfectly valid. What will this print?

In [ ]:
user_text = "42"

try:
    number = int(user_text)
    print("Double:", numbr * 2)
except:
    print("Please type a number.")

It blames the user, but the real problem is a **typo** (`numbr`). A bare `except:` catches **every** error,
including our own bugs, and hides them. Name the exception you expect, and the typo becomes visible:

In [ ]:
try:
    number = int(user_text)
    print("Double:", numbr * 2)
except ValueError:
    print("Please type a number.")

Rules:
1. Always name the exception: `except ValueError:`, not `except:`.
2. Keep the `try` block small: only the risky line(s).
3. Handle the error **usefully** (message, retry, default value). Never hide it silently.

## 8. Real-world example: a safe file loader

`data/marks.txt` is messy: a word instead of a number, a missing mark, a missing comma, a mark of 105, a blank line.
One function checks **one line** and raises; the loader decides to **skip and report**.

In [ ]:
from pathlib import Path


def parse_mark_line(line):
    """Turn 'name,mark' into (name, mark). Raise ValueError if the line is not valid."""
    parts = line.split(",")
    if len(parts) != 2:
        raise ValueError(f"expected 'name,mark' but got '{line}'")
    name = parts[0].strip()
    mark = int(parts[1])
    if mark < 0 or mark > 100:
        raise ValueError(f"mark must be 0-100, got {mark}")
    return name, mark

In [ ]:
def load_marks(path):
    """Return a dict name -> mark from a file. Bad lines are reported and skipped."""
    try:
        with open(path, encoding="utf-8") as marks_file:
            lines = [line.strip() for line in marks_file]
    except FileNotFoundError:
        print(f"Warning: {path} not found. Starting with no marks.")
        return {}

    marks = {}
    for line_number, line in enumerate(lines, start=1):
        if line == "":
            continue
        try:
            name, mark = parse_mark_line(line)
        except ValueError as error:
            print(f"Skipping line {line_number}: {error}")
        else:
            marks[name] = mark
    return marks

In [ ]:
marks = load_marks(Path("data") / "marks.txt")
print(marks)

In [ ]:
print(load_marks(Path("data") / "marks_2025.txt"))

The program never crashes, and every skipped line is reported **with its line number**, so a person can fix the
file.

## Recap

1. `try:` the risky code, `except SomeError:` handle that error; the program continues.
2. Catch **specific** exceptions (`ValueError`, `ZeroDivisionError`, `KeyError`, `FileNotFoundError`); `as error` gives the message.
3. `else` runs when nothing failed; `finally` runs always.
4. Retry loop: `while True` + `try: return float(input(...))` + `except ValueError: print(...)`.
5. `raise ValueError("clear message")` in your own functions; let the caller decide what to do.
6. Never use a bare `except:`. It hides your own bugs.

**Next week:** modules and imports (`from pathlib import Path` will finally make complete sense), testing, and the
Expense Tracker project.